# How the data problems were found

This walks through each problem in the order it turned up, using the saved results in `data/mart`, so it runs without ClickHouse. The queries that produced them are in `sql/`.

The habit throughout: before believing a number, ask what it would look like if the data were wrong, then look for that.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
MART = ROOT / "data" / "mart"
findings = json.loads((MART / "findings.json").read_text(encoding="utf8"))
issues = pd.read_csv(ROOT / "docs" / "issue_log.csv", keep_default_na=False)
overall = issues[issues.station == ""].set_index("check")
pd.set_option("display.max_colwidth", 120, "display.width", 160)

## 1. The row count

The plan said 2,679,264,563 rows, one per station per day. A Parquet file records its row count in its footer, which can be read over HTTP without downloading the file. It says 1,074,089,837. ClickHouse's documentation gives the larger figure as the measurement count before pivoting into rows, so 2.68B is measurements and 1.07B is station-days.

In [2]:
scale = findings["scale"]
print(f"Rows in the file:          {scale['rows']:>13,}")
print(f"Measurements before pivot: {scale['measurements_before_pivot']:>13,}")
print(f"Measurements per row:      {scale['measurements_before_pivot'] / scale['rows']:>13.1f}")
print(f"Last date in the file:     {scale['last_date']:>13}")

Rows in the file:          1,074,089,837
Measurements before pivot: 2,679,264,563
Measurements per row:                2.5
Last date in the file:        2022-09-07


## 2. Zero means missing

The footer also says every column is required, so nothing can be NULL. Then what does a station with no thermometer store? If missing readings are 0, exactly 0.0 C should be far more common than its neighbours, -0.1 C and +0.1 C.

In [3]:
print(overall.loc["DQ01", "detail"])

India: tempMax = 0 on 60207581 rows, against 1 at +0.1 C and 0 at -0.1 C


That settles it. A rollup that divides `sum(tempMax)` by `count()`, as the plan's did, would drag every mean toward zero, so the rollup counts readings separately from rows.

Rain has the same problem in a harder form, because a real dry day is also 0. NOAA's raw files omit readings that were never taken, so they show how many zeros are false.

In [4]:
print(overall.loc["DQ02", "detail"])

Rows on temperature stations 1982-2021: 1136483. NOAA rain reports on those rows: 568808. The difference is rows where 0 rain was never reported.


## 3. Temperature starts in 1973, and rain nearly stops

A file described as 1900-2022 suggests 122 years of everything. Counting stations each year shows something else.

In [5]:
coverage = pd.read_csv(MART / "coverage_by_year.csv").set_index("year")
coverage.loc[[1901, 1950, 1960, 1970, 1973, 1980, 1990, 2000, 2010, 2020], ["stations_rain", "stations_tmax", "stations_full_year", "stations_full_hot_season"]]

,stations_rain,stations_tmax,stations_full_year,stations_full_hot_season
year,,,,
1901,1674,0,0,0
1950,2456,0,0,0
1960,3058,13,0,0
1970,2426,0,0,0
1973,82,85,0,4
1980,110,73,11,41
1990,71,72,5,22
2000,67,71,26,39
2010,78,84,58,73


The plan's 1961-1990 baseline therefore cannot be built. It was replaced with 1991-2020, which IMD itself now uses, and where the most stations have complete seasons.

In [6]:
passing = pd.read_csv(MART / "passing_stations.csv", keep_default_na=False)
pd.Series({
    "Indian stations in the file": findings["quality"]["indian_stations_in_file"],
    "with any temperature": len(passing),
    "with normals (15 complete hot seasons in 1991-2020)": int(passing.has_normals.sum()),
    "in the heatwave panel (also complete in every period)": int(passing.in_heat_panel.sum()),
})

Indian stations in the file                              3806
with any temperature                                      175
with normals (15 complete hot seasons in 1991-2020)        58
in the heatwave panel (also complete in every period)      28
dtype: int64

## 4. Readings that repeat

While checking the hottest readings against published records, the values matched exactly: Churu's 50.8 C on 1 June 2019 and 50.5 C on 28 May 2024 are both documented by IMD. But the next day held the identical value both times. Is that common? A maximum identical to the day before, to a tenth of a degree, should be rare and steady over time.

In [7]:
repeats = pd.read_csv(MART / "repeats_by_year.csv").set_index("year")
repeats.loc[[1985, 1990, 1995, 2000, 2005, 2008, 2009, 2010, 2011, 2012, 2013, 2015], ["stations", "repeat_pct"]].round(1)

,stations,repeat_pct
year,,
1985,28,4.2
1990,22,5.3
1995,41,6.7
2000,38,6.7
2005,57,8.7
2008,56,9.2
2009,56,8.5
2010,54,8.5
2011,55,18.4


It more than doubles between 2010 and 2011, which weather cannot do. My first conclusion was that about half the rise in heatwave days was artefact, because removing every repeat cuts the rise from 64% to 31%, with an interval that includes zero. That conclusion was too strong, and the next two steps show why.

**A prediction that failed.** If a day's peak were carried forward in time, the reading after a repeated pair should be lower more often than the reading before it. It is not: the two rates match. That ruled out the simple version, and prompted a look at individual stations.

**The problem is not spread evenly.** Each station's repeat share either stays near a natural level or jumps by 20 points or more.

In [8]:
groups = pd.read_csv(MART / "station_repeat_groups.csv")
comparable = groups[groups.repeat_group != "unknown"]
jump = comparable["share_2011-15"] - comparable["share_2001-10"]
print("Stations by size of jump, 2001-10 to 2011-15, in points")
print(pd.cut(jump, [-40, 0, 5, 15, 30, 50]).value_counts().sort_index().to_string())
print()
print(comparable.group_label.value_counts().to_string())

Stations by size of jump, 2001-10 to 2011-15, in points
(-40, 0]    18
(0, 5]      38
(5, 15]      0
(15, 30]    22
(30, 50]     1

group_label
Never affected         41
Stepped at 2011        23
Stepped around 1990    14
Other                   1


**And it follows regional lines.** Weather does not stop at a state border.

In [9]:
pd.read_csv(MART / "repeat_groups_by_state.csv", keep_default_na=False).set_index("state")

,clean,jumped,early,other
state,,,,
Andaman & Nicobar Island,1,0,0,0
Andhra Pradesh,0,7,0,0
Assam,4,0,0,0
Bihar,3,0,0,0
Chhattisgarh,2,0,0,0
Goa,0,0,0,1
Gujarat,5,0,0,0
Haryana,0,0,1,0
Jammu & Kashmir,0,0,1,0


**It starts and stops within weeks,** and minimums at the same stations and dates never move.

In [10]:
months = pd.read_csv(MART / "repeat_months_jumped.csv")
window = months[((months.year == 2010) & (months.month >= 8)) | ((months.year == 2011) & (months.month <= 2)) | ((months.year == 2015) & (months.month >= 11)) | ((months.year == 2016) & (months.month <= 4))]
window[["year", "month", "tmax_days", "tmax_share", "tmin_share"]].round(1)

,year,month,tmax_days,tmax_share,tmin_share
19,2010,8,684,6.1,7.2
20,2010,9,660,11.1,5.0
21,2010,10,676,9.8,5.8
22,2010,11,652,14.1,6.8
23,2010,12,692,30.6,4.9
24,2011,1,701,31.4,4.9
25,2011,2,633,32.9,2.8
82,2015,11,663,30.0,3.7
83,2015,12,663,31.8,3.8
84,2016,1,567,20.1,6.8


**Repeats sit on peaks.** A pair of equal readings is a local peak if the days on either side are both lower. Natural pairs fall about evenly between peaks and troughs. A window that catches the previous day's peak piles them up on peaks.

In [11]:
pd.read_csv(MART / "repeat_peak_ratios.csv")[["reading", "repeat_group", "pairs", "peak_pct", "trough_pct", "peaks_per_trough"]].round(1)

,reading,repeat_group,pairs,peak_pct,trough_pct,peaks_per_trough
0,maximum,clean,2471,26.9,20.4,1.3
1,maximum,early,5647,49.1,8.0,6.2
2,maximum,jumped,11791,43.6,7.7,5.7
3,minimum,clean,1536,25.2,22.1,1.1
4,minimum,early,600,20.2,21.2,1.0
5,minimum,jumped,1304,25.3,20.8,1.2


**So the fairest test is the stations it never touched.** Removing every repeat over-corrects, because it also drops genuine hot days. Comparing groups avoids that.

In [12]:
q1 = findings["q1_heatwave_days"]
rows = {"All stations, as recorded": q1["recorded"], "All stations, repeats removed": q1["without_repeats"]}
for key, label in (("clean", "Never affected"), ("jumped", "Stepped at 2011"), ("early", "Stepped around 1990")):
    rows[label + ", as recorded"] = q1["by_repeat_group"][key]["recorded"]
    rows[label + ", repeats removed"] = q1["by_repeat_group"][key]["without_repeats"]
pd.DataFrame(rows).T[["stations", "before", "after", "change_pct", "change_pct_ci", "stations_rising", "stations_falling"]]

,stations,before,after,change_pct,change_pct_ci,stations_rising,stations_falling
"All stations, as recorded",28,1.53,2.52,64.0,"[15.0, 110.0]",16,8
"All stations, repeats removed",28,1.49,1.96,31.0,"[-6.0, 70.0]",15,9
"Never affected, as recorded",10,1.24,1.92,55.0,"[1.0, 107.0]",6,4
"Never affected, repeats removed",10,1.18,1.88,59.0,"[7.0, 115.0]",6,4
"Stepped at 2011, as recorded",14,1.25,1.83,46.0,"[-49.0, 145.0]",6,4
"Stepped at 2011, repeats removed",14,1.25,1.33,6.0,"[-63.0, 83.0]",5,5
"Stepped around 1990, as recorded",3,4.13,8.29,101.0,"[75.0, 266.0]",3,0
"Stepped around 1990, repeats removed",3,3.88,5.48,41.0,"[20.0, 162.0]",3,0


At never-affected stations, removing repeats changes almost nothing, as it should. They rose about 55%, and IMD's own count at 103 stations rose 45%. So heatwave days probably did rise, by about a third to a half, and the record overstates it where the artefact applies.

It also changes another answer. At never-affected stations, days and nights warm at the same pace.

In [13]:
pd.DataFrame(findings["q2_days_and_nights"]["by_repeat_group"]).T

,stations,median_tmax_rate,median_tmin_rate,nights_faster,median_rate
clean,13.0,0.275,0.265,8.0,0.268
jumped,12.0,0.313,0.193,2.0,0.244
early,5.0,0.095,-0.094,1.0,0.003


In [14]:
sensitivity = pd.read_csv(MART / "sensitivity.csv")
sensitivity[["scenario", "stations", "change_recorded_pct", "change_no_repeat_pct"]]

,scenario,stations,change_recorded_pct,change_no_repeat_pct
0,This project's rule,28,64,31
1,"Normals from 1981-2010, not 1991-2020",28,72,36
2,Normals not smoothed across 15 days,28,55,21
3,"Hills from 600 m, not 1,000 m",28,66,33
4,"Coast within 25 km, not 50 km",28,67,34
5,"Coast within 100 km, not 50 km",28,63,30
6,"The plan's simpler rule: 40 C and 4.5 C above normal, everywhere",28,79,38
7,"Counted only in runs of 2 or more days, as IMD declares",28,147,92
8,"Panel needs 2 seasons per period, not 3",33,71,39
9,"Panel needs 4 seasons per period, not 3",20,85,44


## 5. Whole degrees before 1982

Every maximum from 1973 to 1981 is a whole degree, and then the share drops to about 15%. Rounding does not bias a mean, but it does move values across a cut-off: a true 39.6 C becomes 40 C. So the threshold questions start in 1982.

In [15]:
print(overall.loc["DQ11", "detail"])

Whole-degree share of maximums: 100% in 1973-81, 15.6% in 1982-2021


## 6. Rain that halves and recovers

Comparing the same stations across NOAA's sources is a simple water-balance test. A monsoon does not fall by over a third and then return, so this is a change in reporting.

In [16]:
pd.read_csv(MART / "q4_water_balance.csv")[["era", "stations", "season_mm", "rainy_days", "heavy_days", "reports"]].round(1)

,era,stations,season_mm,rainy_days,heavy_days,reports
0,1901-72,70,917.1,42.6,0.8,118.9
1,1973-81,70,501.4,21.6,0.6,71.7
2,1982-98,70,566.2,23.9,0.6,77.0
3,1999-2005,70,605.3,26.9,0.6,70.5
4,2006-15,70,833.2,35.2,1.0,68.9
5,2016-24,70,831.7,35.3,0.9,70.0


A naive trend across these eras would show extreme rain rising sharply from the 1990s. That is the finding this project chose not to report. The only long comparison it trusts is 1901-1970, on complete gauges.

In [17]:
pd.read_csv(MART / "q4_gauge_decades.csv").round(2)

,decade_start,stations,seasons,rainy_days,heavy_days,season_mm
0,1901,1688,15931,38.59,0.79,841.37
1,1911,1714,16982,38.41,0.78,841.18
2,1921,1714,16964,39.01,0.86,868.40
3,1931,1714,17038,39.77,0.81,876.68
4,1941,1714,16547,39.78,0.88,894.38
5,1951,1714,16269,38.56,0.89,877.64
6,1961,1674,12867,36.32,0.84,838.25


## 7. A rule that left India out

Warming rates first required whole years. That left India with no stations at all in the global comparison, because minimum temperatures are sparse before 2000. Each month is already compared with that station's own usual month, so the seasons are removed and a line can go through every qualifying month. The same rule applies to every country.

In [18]:
countries = pd.read_csv(MART / "q6_country_summary.csv", keep_default_na=False)
countries[countries.country.isin(["IN", "US", "CH", "RS", "AS"]) | (countries["rank"] <= 3)][["rank", "name", "stations", "median_rate"]].round(3)

,rank,name,stations,median_rate
0,1,Greenland,7,0.642
1,2,Ukraine,20,0.636
2,3,Turkey,12,0.559
19,20,Russia,489,0.395
22,23,China,178,0.373
32,33,India,31,0.245
34,35,Australia,305,0.235
35,36,United States,4666,0.234


## 8. Where I was wrong in the benchmark

Two more of this project's expectations failed, and both are worth keeping.

- **The plan's `substring` filter seemed to cost almost nothing, and that was wrong.** It nearly matched `LIKE 'IN%'` in the first benchmark, but `EXPLAIN` showed it never used the sort key. ClickHouse's query condition cache was remembering which parts of the table matched, so the "cold" run was not cold. With that cache cleared, it reads all 1.07B rows and takes more than twice as long. The harness now clears it.
- **The first version of Q6 lost to DuckDB, even on the rollup.** The fault was in the SQL: its join ran the months step twice. A window function and in-order aggregation made it about three times faster.

In [19]:
bench = pd.read_csv(MART / "benchmark.csv")
bench.pivot_table(index="question", columns=["run", "engine"], values="seconds")

run                                                                 cold                                     median_warm                                 
engine                                                    clickhouse_raw clickhouse_rollup duckdb_parquet clickhouse_raw clickhouse_rollup duckdb_parquet
question                                                                                                                                                 
Every country by decade, full scan                                16.593             0.253         13.056         14.905             0.129         12.357
India, every station-year, by key prefix                           1.377             0.263          0.975          1.080             0.115          0.591
India, same result, plan's substring filter                        2.972               NaN            NaN          1.080               NaN            NaN
One station's whole history                                        0.369             0.220          0.404          0.083             0.073          0.117
Q6, first version: a join that runs the months step twice         33.451             7.674          6.790         31.865             7.335          6.738
Q6, rewritten: window function, in key order                      24.127             2.530          7.552         20.152             2.337          6.585

In [20]:
reads = pd.read_csv(MART / "benchmark_reads.csv")
reads.assign(million_rows=(reads.read_rows / 1e6).round(1), gb_read=(reads.read_bytes / 1e9).round(2))[["query", "million_rows", "gb_read"]]

,query,million_rows,gb_read
0,b1_india_station_years.raw.sql,61.7,0.61
1,b1_india_station_years.rollup.sql,0.2,0.00
2,b2_india_filter_no_index.raw.sql,1074.1,2.61
3,b3_one_station_history.raw.sql,0.5,0.00
4,b3_one_station_history.rollup.sql,0.0,0.00
5,b4_country_decade_scan.raw.sql,1074.1,10.70
6,b4_country_decade_scan.rollup.sql,3.3,0.08
7,b5_q6_country_rates.raw.sql,947.5,12.55
8,b5_q6_country_rates.rollup.sql,18.0,0.52
9,b6_q6_rewritten.raw.sql,473.8,6.50


## What the checks add up to

Every check is in `docs/issue_log.csv`, and the departures from the original plan are in `docs/plan_corrections.md`.

In [21]:
issues.groupby(["check", "severity"]).agg(rows=("count", "size"), total=("count", "sum"), what=("description", "first")).reset_index()

,check,severity,rows,total,what
0,DQ01,high,1,60207581,"Missing readings are stored as 0. If so, exactly 0 is far more common than its neighbours, -0.1 and +0.1 C."
1,DQ02,high,1,567675,A zero in precipitation can mean dry or not reported. Compare the Parquet's zeros with NOAA's raw file for temperatu...
2,DQ03,low,1,0,"Duplicates: more than one row per station per date, for every Indian station."
3,DQ04,medium,1,0,Plausibility: tempMin above tempMax on the same day.
4,DQ05,low,1,0,"Plausibility: maximum or minimum outside -10 to 52 C at stations below 1,000 m."
5,DQ06,info,1,0,"Plausibility: negative rainfall, and any single day above India's 24-hour record of about 1,560 mm."
6,DQ07,medium,37,124965,"Gaps: longest run of days with no temperature reading, per station, between its first and last reading since 1982."
7,DQ08,low,47,47,Location: IN-prefix stations whose coordinates fall outside the outline of India.
8,DQ09,low,1,0,Location: the Parquet's coordinates disagree with NOAA's current station list by more than 0.1 degree.
9,DQ10,high,1,91724,"Carried-over peaks: maximum identical to the day before, per year. The share climbs from a natural level near 4.5%."
